# Task 2 — Pose estimation benchmarking on your squat videos

Goal for this step:
1. Apply pose estimators to extract **17 COCO-ordered joints** per sampled frame.
2. Compare models on **speed** and a proxy for **accuracy** based on how well the detected knee angle differs between annotated error vs non-error time windows.

This notebook benchmarks:
- YOLOv11 pose (via `ultralytics`)
- torchvision Keypoint R-CNN (17 COCO keypoints)

Note: MediaPipe BlazePose (tasks API) may fail in this runtime due to missing system libraries (e.g. `libEGL.so.1`). If you enable it later, you can plug it into the same benchmark loop.
Important note: since your repo does not include ground-truth keypoints, “accuracy” here is measured as a **downstream agreement score** with your provided knee-error intervals.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import List, Tuple

import numpy as np

PROJECT_ROOT = Path('/workspace')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fitness_adapt.io_utils import load_json
from fitness_adapt.pose import (
    extract_yolo11_pose_coco17,
    extract_torchvision_keypointrcnn_coco17,
)
from fitness_adapt.evaluation import knee_angle_separation_score, extraction_speed_fps
from fitness_adapt.project import ProjectPaths

paths = ProjectPaths.from_root(PROJECT_ROOT)
VIDEO_DIR = paths.squat_video_dir

train_keys = load_json(paths.split_path('train'))
error_fwd = load_json(paths.error_knees_forward_path)
error_inward = load_json(paths.error_knees_inward_path)

Interval = Tuple[float, float]

def union_intervals(raw: List[List[float]]) -> List[Interval]:
    if not raw:
        return []
    intervals = [(float(a), float(b)) for a, b in raw]
    intervals = sorted(intervals, key=lambda x: x[0])
    merged: List[Interval] = [intervals[0]]
    for s, e in intervals[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged

def get_union_error_intervals(video_key: str) -> List[Interval]:
    fwd = union_intervals(error_fwd.get(video_key, []))
    inward = union_intervals(error_inward.get(video_key, []))
    return union_intervals([[s, e] for s, e in (fwd + inward)])


In [ ]:
# Benchmark configuration (keep small for quick iteration)
sample_keys = train_keys[:4]

frame_stride = 4
max_frames = 160
conf_threshold = 0.35

models = [
    ('yolo11n_pose', dict(frame_stride=frame_stride, max_frames=max_frames, conf_threshold=conf_threshold, yolo_weights='yolo11n-pose.pt', device='cpu')),
    ('keypointrcnn_pose', dict(frame_stride=frame_stride, max_frames=max_frames, conf_threshold=conf_threshold, device='cpu')),
]

results = {}

for model_name, kwargs in models:
    model_results = []
    print(f"\n== {model_name} ==")
    for k in sample_keys:
        video_path = str(VIDEO_DIR / f'{k}.mp4')
        error_intervals = get_union_error_intervals(k)

        if model_name == 'yolo11n_pose':
            out = extract_yolo11_pose_coco17(video_path, **kwargs)
        else:
            out = extract_torchvision_keypointrcnn_coco17(video_path, **kwargs)

        fps = extraction_speed_fps(out.extraction_time_sec, out.keypoints_xy.shape[0])
        sep = knee_angle_separation_score(out.keypoints_xy, out.times_sec, error_intervals)
        model_results.append((k, fps, sep, float(out.extraction_time_sec)))
        print(f"{k}: extracted_frames={out.keypoints_xy.shape[0]} speed_fps={fps:.2f} separation={sep:.3f} time_sec={out.extraction_time_sec:.2f}")

    fps_vals = [r[1] for r in model_results]
    sep_vals = [r[2] for r in model_results]
    results[model_name] = {
        'mean_speed_fps': float(np.mean(fps_vals)) if fps_vals else 0.0,
        'mean_knee_separation': float(np.mean(sep_vals)) if sep_vals else 0.0,
        'per_key': model_results,
    }

print("\nSummary:")
for m in results:
    print(m, results[m]['mean_speed_fps'], results[m]['mean_knee_separation'])


## Selecting the best model for your use case

Use two numbers from the output:
- `mean_speed_fps`: higher is better (for real-time overlay + feedback)
- `mean_knee_separation`: higher indicates the detected knee angle aligns better with the annotated error windows

After you pick the best model, you’ll reuse it in Task 3–6 for keypoint preprocessing + feature extraction + sequence model training.
